# HNSW Algorithm Implementation

## **Components**
#### - Layer Assignment - building the HNSW structure
#### - Search - method for ANN search


## **Search**

#### Suppose we have n nodes (A-E) in a layer, *query point Q*, and we want the 2 NNs (ef=2). We have the name of each node, their distances to Q, and their graph connections (who is connected to who at this layer). One of the n nodes, A, will be out *entry point*; we start here.

#### * *min-heap* : For *candidates* keeps the smallest value at the top, index 0 = smallest dist --> explore next
#### * *max-heap* : For *found*, keeps the greatest value at the top (as a negative), index 0 = worst result (greatest distance), first to evict

#### We start a loop wherein we:
#### (1) Pop the smallest valued node, A, from *candidates*, the node we explore from. 
#### (2a) If this value is NOT worse than the *worst_found* (the largest value in *found*), then continue. 
#### (2b) If this value is worse than the *worst_found* , discared the node.

#### Assuming (2a), and noting that (i) the first node has been popped out of *candidates* while (ii) staying in *found*, we check out A's neighbors, assuming B and C are designated as such. First we visit B: say the dist(Q,B)=2.0 < *worst_found* (5.0) and push B onto both heaps. 

#### Following (2a) again, we continue to A's other neighbor, C. Suppose dist(Q,C)=8.0 > *worst_found*(5.0). Since *found* is already full, C is discarded entirely.

#### *Now* we start the loop all over again! We pop B from *candidates*, compare with *worst_found* (still A), and explore B's neighbors, which could be [A, D, E]. Since A is already visited, we skip it and visit D.

#### Now that we've done this in detail, I'll be less verbose here on out.
#### dist(Q,D)=1.0 < *worst_found* (5.0) --> push D onto both heaps. *found* has 3 items but we pop the worst (A at 5.0) because ef=2.

#### Suppose dist(Q,E)=3.0 < *worst_found* (now 2.0, B). 3.0>2.0, so we discard E.

#### Loop again! Pop smallest from *candidates* (D at 1.0), 1.0 < *worst_found* (2.0), so we explore D's neighbors, say [B, E]. Both visited previously, skip both. Nothing pushed.

#### Candidates now empty, loop ends. Return *found*: [(1.0. D), (2.0, B)]. These are our 2 ANNs.

#### Generally: We consider a node (as a *candidate*), go through its neighbors, and continually update *found* until we have 2 nodes. Then we try continue to the next node, and so on.

In [ ]:
import heapq

: 

In [ ]:
def search_layer(self, query, entry_points, ef, layer):
    visited = set(entry_points)
    candidates = []
    found = []

    for ep in entry_points:
        dist = self._dist(query, ep)
        heapq.heapppush(candidates, (dist, ep)) # push ep to candidates/min-heap w// distance
        heapq.heappush(found, (-dist, ep)) # push ep to found/max-heap w// distance
    
    while candidates:
        c_dist, c_id = heapq.heappop(candidates)
        worst_found = -found[0][0] # worst distance in found/max-heap
        
        if c_dist > worst_found:
            break
        
        for neighbor_id in self.graphs[layer].get(c_id, []):
            if neighbor_id in visited: 
                continue
            visited.add(neighbor_id)

            n_dist = self.graphs[layer].dist(query, self.vectors[neighbor_id])
            worst_found = -found[0][0]

            if n_dist < worst_found or len(found) < ef:
                heapq.heappush(candidates, (n_dist, neighbor_id))
                heapq.heappush(found, (-n_dist, neighbor_id))
                if len(found) > ef:
                    heapq.heappop(found)

    return sorted((-d, nid) for d, nid in found)

                
        



: 